# 03 — Simulation & Evaluation
Run closed-loop MPC in MuJoCo with the trained dynamics model.

**Inputs:** `experiments/weights/<run_name>.pt`, `assets/hopper.xml`  
**Outputs:** `experiments/videos/`, `experiments/figures/`

**Run after:** `02_training.ipynb`

## 1. Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

from hopper import HopperMLP, run_simulation
from hopper.evaluation import plot_control_forces
from hopper.mpc import set_normalization_stats
from configs.mpc_config import load_config, print_config

np.random.seed(42)
torch.manual_seed(42)

## 2. Configuration

In [ ]:
RUN_NAME    = "hopper_mlp_v1"  # Must match what was saved in 02_training.ipynb

WEIGHTS_DIR = "../experiments/weights"
VIDEO_DIR   = "../experiments/videos"
FIGURE_DIR  = "../experiments/figures"
HOPPER_XML  = "../assets/hopper.xml"

MODEL_WEIGHTS = os.path.join(WEIGHTS_DIR, f"{RUN_NAME}.pt")
NORM_STATS    = os.path.join(WEIGHTS_DIR, f"{RUN_NAME}_norm.npz")

for d in [VIDEO_DIR, FIGURE_DIR]:
    os.makedirs(d, exist_ok=True)

# Check required files exist
assert os.path.exists(MODEL_WEIGHTS), f"Weights not found: {MODEL_WEIGHTS} — run 02_training.ipynb first"
assert os.path.exists(NORM_STATS),    f"Norm stats not found: {NORM_STATS}   — run 02_training.ipynb first"
assert os.path.exists(HOPPER_XML),    f"XML not found: {HOPPER_XML}"

print(f"✓ Weights:   {MODEL_WEIGHTS}")
print(f"✓ Norm stats: {NORM_STATS}")
print(f"✓ XML:        {HOPPER_XML}")

## 3. Load Normalization Stats
Must be loaded before simulation so MPC normalizes inputs correctly.

In [ ]:
stats = np.load(NORM_STATS)
set_normalization_stats(
    stats['X_mean'], stats['X_std'],
    stats['Y_mean'], stats['Y_std'],
)
print("✓ Normalization stats loaded")

## 4. Single Simulation Run

In [ ]:
cfg = load_config('hopping')
print_config('hopping')

results = run_simulation(
    hopper_xml=HOPPER_XML,
    model_weights=MODEL_WEIGHTS,
    sim_time=5.0,
    fps=80,
    z_des=cfg['Z_DES'],
    output_video=os.path.join(VIDEO_DIR, f"{RUN_NAME}_hopping.mp4"),
    device='cpu',
    verbose=True,
    plot=False,   # We'll make our own plots below
)

## 5. Height Tracking Analysis

In [ ]:
torso  = results['torso_com']
system = results['system_com']

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(torso['t'],  torso['z'],  label='Torso height',  linewidth=2)
ax.plot(system['t'], system['z'], '--', label='System COM height', linewidth=2, alpha=0.7)
ax.axhline(cfg['Z_DES'], color='r', linestyle=':', linewidth=2,
           label=f"Target ({cfg['Z_DES']*100:.0f} cm)")

ax.set_xlabel('Time [s]')
ax.set_ylabel('Height [m]')
ax.set_title('Height Tracking')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, f"{RUN_NAME}_height.png"), dpi=150, bbox_inches='tight')
plt.show()

# Statistics
z = np.array(torso['z'])
print(f"Height — mean: {z.mean()*100:.2f} cm  "
      f"std: {z.std()*100:.2f} cm  "
      f"min: {z.min()*100:.2f} cm  "
      f"max: {z.max()*100:.2f} cm")

## 6. XY Drift Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Time series
axes[0].plot(torso['t'], np.array(torso['x'])*100, label='X')
axes[0].plot(torso['t'], np.array(torso['y'])*100, label='Y')
axes[0].set_xlabel('Time [s]')
axes[0].set_ylabel('Position [cm]')
axes[0].set_title('Lateral Drift Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Top-down view
axes[1].plot(np.array(torso['x'])*100, np.array(torso['y'])*100, alpha=0.7)
axes[1].scatter([torso['x'][0]*100],  [torso['y'][0]*100],  c='green', s=80, zorder=5, label='Start')
axes[1].scatter([torso['x'][-1]*100], [torso['y'][-1]*100], c='red',   s=80, zorder=5, label='End')
axes[1].set_xlabel('X [cm]')
axes[1].set_ylabel('Y [cm]')
axes[1].set_title('Top-Down Trajectory')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, f"{RUN_NAME}_drift.png"), dpi=150, bbox_inches='tight')
plt.show()

## 7. Control Force Analysis

In [ ]:
plot_control_forces(results['controls'], dt=1/80.0)

## 8. Config Comparison
Run multiple configurations and compare performance side-by-side.

In [ ]:
CONFIG_NAMES = ['conservative', 'hopping', 'high_jump']
all_results  = {}

for config_name in CONFIG_NAMES:
    print(f"\nRunning: {config_name}")
    cfg = load_config(config_name)

    r = run_simulation(
        hopper_xml=HOPPER_XML,
        model_weights=MODEL_WEIGHTS,
        sim_time=3.0,
        fps=80,
        z_des=cfg['Z_DES'],
        output_video=os.path.join(VIDEO_DIR, f"{RUN_NAME}_{config_name}.mp4"),
        device='cpu',
        verbose=False,
        plot=False,
    )
    all_results[config_name] = (cfg, r)

print("\nDone.")

In [ ]:
# Side-by-side height plot
fig, ax = plt.subplots(figsize=(12, 4))

colors = ['steelblue', 'darkorange', 'seagreen']
for (config_name, (cfg, r)), color in zip(all_results.items(), colors):
    z = np.array(r['torso_com']['z'])
    t = r['torso_com']['t']
    ax.plot(t, z * 100, label=config_name, color=color, linewidth=2)
    ax.axhline(cfg['Z_DES'] * 100, color=color, linestyle=':', alpha=0.5)

ax.set_xlabel('Time [s]')
ax.set_ylabel('Height [cm]')
ax.set_title('Config Comparison — Height Tracking')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, f"{RUN_NAME}_config_comparison.png"), dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f"\n{'Config':<15} {'Target (cm)':>11} {'Mean z (cm)':>11} {'Std z (cm)':>10} {'Max z (cm)':>10}")
print("-" * 60)
for config_name, (cfg, r) in all_results.items():
    z = np.array(r['torso_com']['z']) * 100
    print(f"{config_name:<15} {cfg['Z_DES']*100:>11.1f} {z.mean():>11.2f} {z.std():>10.2f} {z.max():>10.2f}")